# Leakage-aware Evaluation of Infant Cry Classification — Colab runner

Runs the three core experiments of the paper (`multiclass`, `binary`, `leakage`) from Google Colab, with data and repo living on Google Drive. See the note at the bottom on which of these actually use a GPU when one is available — it's not all three.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the repo

Two options — pick ONE. Default below is the real path for this project (repo already lives in Drive, inside `Mi unidad/Publicaciones/Chillanto CIARP/`). If you're reusing this notebook for a different Drive layout, adjust `REPO_DIR` or use Option B to clone fresh instead.

In [ ]:
# Option A (default): repo already present in Drive, at its real path for this project.
REPO_DIR = "/content/drive/MyDrive/Publicaciones/Chillanto CIARP/leakage-aware-infant-cry-classification"
%cd "$REPO_DIR"

# Option B: clone fresh into the Colab runtime instead (uncomment and set REPO_URL).
# REPO_URL = "https://github.com/pantrok/leakage-aware-infant-cry-classification.git"
# REPO_DIR = "/content/leakage-aware-infant-cry-classification"
# !git clone "$REPO_URL" "$REPO_DIR"
# %cd "$REPO_DIR"

## 3. Point to the dataset

See [`data/README.md`](../data/README.md) for the expected folder layout (`1s_asphyxia/`, `1s_deaf/`, `1s_hunger/`, `1s_normal/`, `1s_pain/`). The dataset itself is not distributed in this repo.

For this project the real data already lives in Drive, in a sibling folder to the repo (`Bebes Grid indiv - Optuna - 2/Data/`) rather than copied into `data/dataset/` — Option A below points straight at it so nothing has to be duplicated. If you'd rather keep a copy inside `data/dataset/` (e.g. for a different machine/Drive layout), use Option B instead.

In [ ]:
import os

# Option A (default): point directly at the real data folder already in Drive, outside the repo.
DATA_DIR = "/content/drive/MyDrive/Publicaciones/Chillanto CIARP/Bebes Grid indiv - Optuna - 2/Data"

# Option B: use a copy placed inside the repo instead (uncomment).
# DATA_DIR = os.path.join(REPO_DIR, "data", "dataset")

expected = ["1s_asphyxia", "1s_deaf", "1s_hunger", "1s_normal", "1s_pain"]
missing = [d for d in expected if not os.path.isdir(os.path.join(DATA_DIR, d))]
assert not missing, (
    f"DATA_DIR '{DATA_DIR}' is missing expected subfolders: {missing}. "
    "Place the Baby Chillanto 1s_* folders there before continuing "
    "(see data/README.md)."
)
print("Dataset layout OK:", DATA_DIR)

## 4. Install dependencies

In [ ]:
%pip install -r requirements.txt

## 5. (Recommended) Sanity check: recording-ID grouping

Confirms the file-name based grouping used by the `leakage` mode's `G-1s` condition is working as expected before committing to a long run. Should print `M (grabaciones distintas, total) = 73`.

In [ ]:
!python scripts/count_recordings.py --data_dir "$DATA_DIR"

## 6. Run the three experiments

Flags shown are the scripts' own defaults (100 Optuna trials, 5 seeds, 5-fold CV, `k_best_d=60`, `fv_n_components=16`) — the same ones used to produce `results/*.csv` in this repo. Lower `--n_trials`/`--n_seeds` for a quick smoke test before committing to a full run.

**Read the GPU note at the bottom before running `leakage` expecting a GPU speed-up — as of this notebook, that script does not use one.**

In [ ]:
# Multiclass (Tables 2-3): feature-set x classifier Optuna search
!python main.py multiclass --data_dir "$DATA_DIR" --n_trials 100 --n_seeds 5 --n_splits 5 --balanced \
    --csv_summary results/optuna_summary_multiclass.csv --csv_history results/optuna_history_multiclass.csv

In [ ]:
# Binary (Table 4): healthy vs pathology
!python main.py binary --data_dir "$DATA_DIR" --n_trials 100 --n_seeds 5 --n_splits 5 --balanced \
    --csv_summary results/binary_summary.csv --csv_history results/binary_history.csv

In [ ]:
# Leakage comparison (Section 4.3, Tables 5-6, Figures 6-8): S-1s vs G-1s.
# This is the expensive one — see the time-budget note below before launching it
# unattended. Run in a cell by itself so Colab doesn't disconnect it as "idle"
# while it's still computing (interact with the tab occasionally, or use a
# Colab Pro background execution session for very long runs).
!python main.py leakage --data_dir "$DATA_DIR" --n_seeds 5 --n_splits 5 --balanced \
    --csv_results results/leakage_results.csv --csv_stats results/leakage_stats.csv --plot_dir results/leakage_plots

## Notes on compute backend and expected runtime

**`multiclass` and `binary`** (`scripts/optuna_search.py`, `scripts/optuna_binary.py`): `src/gpu_classifiers.py` tries `cuML` (RAPIDS) for SVM/kNN and PyTorch+CUDA for the MLP, and **falls back to scikit-learn on CPU automatically** if they aren't available — this CPU/scikit-learn fallback is what reproduces the exact figures reported in the paper. A Colab GPU runtime will use PyTorch for the MLP if CUDA is detected (Colab ships PyTorch already); installing `cuml-cu12` is optional and only speeds up SVM/kNN. `cuML` has no native Windows build (Linux/WSL2 or Colab only).

**`leakage`** (`scripts/leakage_comparison.py`): as of this notebook, this script builds its classifiers directly from `sklearn` (`SVC`, `KNeighborsClassifier`, `MLPClassifier`, `RandomForestClassifier`) and does **not** go through `src/gpu_classifiers.py` — a Colab GPU runtime gives it no speed-up at all today, it runs identically to a CPU-only machine. A single small-scale smoke test (`--feature_sets B --classifiers MLP --n_splits 2 --n_seeds 1`, i.e. a small fraction of a full run) took about 65 minutes on a CPU-only desktop; the full default grid (4 feature sets × 6 classifiers × 2 modes × 5 seeds × 5 folds) is on the order of days at that rate, GPU runtime or not. If you're reading this before that's been changed, budget accordingly or ask about wiring `leakage_comparison.py` through `src/gpu_classifiers.py` first.